# Level 1 – Data Exploration, Preprocessing & Analysis
**Cognifyz Data Science Internship**
- Task 1: Data Exploration and Preprocessing
- Task 2: Descriptive Analysis (with custom Hash Table)
- Task 3: Geospatial Analysis


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import folium
from folium.plugins import MarkerCluster, HeatMap
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings('ignore')

from utils.hash_table import HashTable, build_frequency_table

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

print('All libraries loaded successfully.')


## Task 1 · Data Exploration and Preprocessing

In [ ]:
# ── Load Dataset ────────────────────────────────────────────────────────────
DATA_PATH = '../dataset/Dataset.csv'
df_raw = pd.read_csv(DATA_PATH, encoding='latin-1')
df = df_raw.copy()
print('Dataset loaded:', df.shape)
df.head()


In [ ]:
# ── Shape & Data Types ──────────────────────────────────────────────────────
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)


In [ ]:
# ── Missing Value Analysis ───────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('Columns with missing values:')
print(missing_df.sort_values('Missing %', ascending=False))

# Plot
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['Missing %'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Missing Value Percentage per Column')
    ax.set_xlabel('Missing %')
    plt.tight_layout(); plt.show()


In [ ]:
# ── Advanced Preprocessing ──────────────────────────────────────────────────

# 1. Lowercase text columns + strip spaces
text_cols = df.select_dtypes(include='object').columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

# 2. Replace 'nan' strings back to actual NaN
df.replace('nan', np.nan, inplace=True)

# 3. Median imputation for numeric columns
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f'  Numeric  | {col:40s} | filled with median = {median_val:.4f}')

# 4. Mode imputation for categorical columns (except Cuisines)
cat_cols = [c for c in text_cols if c.lower() != 'cuisines']
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f'  Categoric| {col:40s} | filled with mode = {mode_val}')

# 5. Replace missing cuisines with 'unknown'
df['Cuisines'] = df['Cuisines'].fillna('unknown')

print('\nMissing values after imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('(None means fully imputed)')


In [ ]:
# ── Outlier Handling (IQR Method) ────────────────────────────────────────────

def iqr_bounds(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outlier_cols = ['Votes', 'Average Cost for two', 'Aggregate rating']
outlier_cols = [c for c in outlier_cols if c in df.columns]

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(15, 5))
for i, col in enumerate(outlier_cols):
    lo, hi = iqr_bounds(df[col])
    n_out  = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col:30s}  lower={lo:.2f}  upper={hi:.2f}  outliers={n_out}')
    # Cap outliers (Winsorize)
    df[col] = df[col].clip(lower=lo, upper=hi)
    axes[i].boxplot(df[col].dropna(), vert=True)
    axes[i].set_title(f'{col}\n(after capping)')

plt.suptitle('Boxplots After IQR-Based Outlier Capping', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── Normalization ─────────────────────────────────────────────────────────────
# Choice: MinMaxScaler
# Reason: Aggregate Rating is bounded [0,5]; tree-based models don't need
#         standardization, but the dashboard slider benefits from [0,1] range.

scaler   = MinMaxScaler()
norm_col = 'Aggregate rating'
if norm_col in df.columns:
    df[norm_col + '_norm'] = scaler.fit_transform(df[[norm_col]])
    print('MinMaxScaler applied to:', norm_col)
    print(df[[norm_col, norm_col + '_norm']].describe())


In [ ]:
# ── Class Imbalance Analysis ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw histogram
axes[0].hist(df['Aggregate rating'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Aggregate Rating')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Rounded rating (0-5) counts
rating_counts = df['Aggregate rating'].round(0).value_counts().sort_index()
axes[1].bar(rating_counts.index.astype(str), rating_counts.values, color='salmon')
axes[1].set_title('Frequency by Rounded Rating (Class Distribution)')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')

plt.tight_layout(); plt.show()

print("\nInsight: Rating 0.0 is heavily over-represented (not-yet-rated restaurants).")
print("This creates class imbalance. During modelling, such records should be")
print("either excluded or handled using stratified splits.")


## Task 2 · Descriptive Analysis

In [ ]:
# ── Statistical Summary ──────────────────────────────────────────────────────
numeric_summary = df.select_dtypes(include=np.number).agg(
    ['mean', 'median', 'std', 'min', 'max']
).T.round(3)
print(numeric_summary)


In [ ]:
# ── Custom Hash Table: City, Cuisine, Country Frequencies ────────────────────

# City frequency
city_ht = build_frequency_table(df['City'], capacity=256)
print('--- City Hash Table ---')
print('Top 10 cities:', city_ht.top_n(10))
print(city_ht.collision_report())

# Cuisine frequency  (multi-valued: 'Italian, Chinese' -> split)
cuisine_ht = build_frequency_table(df['Cuisines'], capacity=512)
print('\n--- Cuisine Hash Table ---')
print('Top 10 cuisines:', cuisine_ht.top_n(10))
print(cuisine_ht.collision_report())

# Country code frequency (numeric key cast to str)
country_ht = build_frequency_table(df['Country Code'].astype(str), capacity=64)
print('\n--- Country Hash Table ---')
print('Top 10 countries:', country_ht.top_n(10))
print(country_ht.collision_report())


In [ ]:
# ── Collision Demonstration ──────────────────────────────────────────────────
# Force a tiny capacity to show chaining in action
demo_ht = HashTable(capacity=4)
test_keys = ['New Delhi', 'Mumbai', 'Bangalore', 'Chennai', 'Hyderabad',
             'Kolkata', 'Pune', 'Ahmedabad']
for k in test_keys:
    demo_ht.increment(k)

print('Demo Hash Table (capacity=4):')
print(demo_ht.collision_report())
print('Stored data:', demo_ht.get_all())

# Search & Delete demo
print('\nSearch Mumbai:', demo_ht.search('Mumbai'))
demo_ht.delete('Pune')
print('After deleting Pune, search Pune:', demo_ht.search('Pune'))


In [ ]:
# ── Top Cities Plot (using hash table results) ───────────────────────────────
top_cities   = city_ht.top_n(15)
cities_df    = pd.DataFrame(top_cities, columns=['City', 'Count'])

fig = px.bar(cities_df, x='Count', y='City', orientation='h',
             title='Top 15 Cities by Restaurant Count',
             color='Count', color_continuous_scale='Blues_r',
             template='plotly_white')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


In [ ]:
# ── Top Cuisines Plot ────────────────────────────────────────────────────────
top_cuisines  = cuisine_ht.top_n(15)
cuisines_df   = pd.DataFrame(top_cuisines, columns=['Cuisine', 'Count'])

fig = px.bar(cuisines_df, x='Count', y='Cuisine', orientation='h',
             title='Top 15 Cuisines by Frequency',
             color='Count', color_continuous_scale='Oranges_r',
             template='plotly_white')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


## Task 3 · Geospatial Analysis

In [ ]:
# ── Filter valid lat/lon ─────────────────────────────────────────────────────
geo_df = df[
    (df['Latitude'].between(-90, 90)) &
    (df['Longitude'].between(-180, 180)) &
    (df['Latitude'] != 0) &
    (df['Longitude'] != 0)
].copy()
print(f'Geo-valid rows: {len(geo_df)} / {len(df)}')


In [ ]:
# ── Restaurant Location Map (Folium + Cluster) ───────────────────────────────
m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')
cluster = MarkerCluster(name='Restaurants').add_to(m)

sample = geo_df.sample(min(2000, len(geo_df)), random_state=42)
for _, row in sample.iterrows():
    popup_html = (
        f"<b>{row.get('Restaurant Name', '')}</b><br>"
        f"City : {row.get('City', '')}<br>"
        f"Rating: {row.get('Aggregate rating', '')}<br>"
        f"Cuisine: {row.get('Cuisines', '')}"
    )
    color = ('green' if row['Aggregate rating'] >= 4
             else 'orange' if row['Aggregate rating'] >= 3
             else 'red')
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(cluster)

folium.LayerControl().add_to(m)
map_path = '../visualizations/restaurant_map.html'
m.save(map_path)
print(f'Map saved to {map_path}')
m


In [ ]:
# ── City Density Heatmap (Folium) ────────────────────────────────────────────
heat_data = geo_df[['Latitude', 'Longitude']].values.tolist()
heat_map  = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB dark_matter')
HeatMap(heat_data, radius=8, blur=10, max_zoom=13).add_to(heat_map)
heatmap_path = '../visualizations/city_density_heatmap.html'
heat_map.save(heatmap_path)
print(f'Heatmap saved to {heatmap_path}')
heat_map


In [ ]:
# ── Rating Distribution by Country (Plotly) ─────────────────────────────────
country_rating = (
    geo_df.groupby('Country Code')['Aggregate rating']
    .mean().reset_index()
    .rename(columns={'Country Code': 'Code', 'Aggregate rating': 'Avg Rating'})
)
fig = px.choropleth(
    country_rating, locations='Code', locationmode='ISO-3166 Numeric',
    color='Avg Rating', hover_name='Code',
    color_continuous_scale='RdYlGn',
    title='Average Restaurant Rating by Country',
    template='plotly_white'
)
fig.show()


In [ ]:
# ── Lat/Lon vs Rating Correlation ────────────────────────────────────────────
corr_geo = geo_df[['Latitude', 'Longitude', 'Aggregate rating']].corr()
print('Correlation Matrix:')
print(corr_geo)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr_geo, annot=True, fmt='.3f', cmap='coolwarm', ax=ax,
            linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Geospatial Correlation Matrix')
plt.tight_layout(); plt.show()


In [ ]:
# ── Scatter: Longitude vs Rating ─────────────────────────────────────────────
fig = px.scatter(
    geo_df.sample(3000, random_state=42),
    x='Longitude', y='Aggregate rating',
    color='Aggregate rating',
    color_continuous_scale='RdYlGn',
    opacity=0.5,
    title='Longitude vs Aggregate Rating',
    template='plotly_white'
)
fig.show()

print("\nInsight: Most high-rated restaurants cluster in certain longitude bands,")
print("reflecting dense urban food cultures in South/Southeast Asia and Europe.")


In [ ]:
# ── Save clean dataframe for re-use by Level 3 ──────────────────────────────
CLEAN_PATH = '../dataset/cleaned_dataset.csv'
df.to_csv(CLEAN_PATH, index=False)
print(f'Clean dataset saved to {CLEAN_PATH}')


## Summary
- Completed Task 1: Preprocessing, outlier handling, normalization, imbalance analysis.
- Completed Task 2: Descriptive stats + Hash Table frequency analysis.
- Completed Task 3: Geospatial maps + correlation analysis.
